# PhysioGraphSleep — Training Pipeline

Single-channel EEG sleep staging with heterogeneous graph neural networks.

**Target**: Sleep-EDF-20 — 20 subjects, 5-class (W/N1/N2/N3/REM)

## Bu notebook'un ilkeleri
- **Tüm dosyalar `physiographsleep/` klasörünün İÇİNDE** üretilir (cache, checkpoints, logs, outputs).
- **WandB** opsiyonel; `WANDB_ENABLED=True` yapıp `WANDB_API_KEY` ortam değişkenini ayarlayın.
- **Live plots** her epoch sonunda otomatik güncellenir (matplotlib).
- **Detaylı tablo**: train/val loss + accuracy + macro-F1 + per-class F1 her epoch için yazdırılır.

## Kullanım
1. Colab → Runtime → Change runtime type → **GPU (T4/L4/A100)**
2. Hücreleri sırayla çalıştırın.


## 1. Setup & Repo Clone

In [ ]:
import os, sys, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*Channels contain different.*")
warnings.filterwarnings("ignore", message=".*Highpass cutoff frequency.*")
warnings.filterwarnings("ignore", message=".*Lowpass cutoff frequency.*")
os.environ["MNE_LOGGING_LEVEL"] = "ERROR"

# === Yapılandırma flagları ===
WANDB_ENABLED = True  # True yapıp aşağıdaki API key cell'ini doldurun
WANDB_PROJECT = "physiographsleep"
WANDB_RUN_NAME = "sleepedf20"

# Colab /dev/shm genişlet (multi-worker DataLoader için)
if "google.colab" in sys.modules:
    !df -h /dev/shm 2>/dev/null
    !sudo mount -o remount,size=2G /dev/shm 2>/dev/null
    !df -h /dev/shm 2>/dev/null


In [ ]:
# Repo clone (Colab) veya local repo path tespiti
REPO_URL = "https://github.com/0nur0duncu/physiographsleep.git"

if "google.colab" in sys.modules:
    PROJECT_DIR = "/content/physiographsleep"
    if os.path.exists(PROJECT_DIR):
        !cd {PROJECT_DIR} && git pull
        print(f"Repo güncellendi: {PROJECT_DIR}")
    else:
        !git clone {REPO_URL} {PROJECT_DIR}
        print(f"Repo klonlandı: {PROJECT_DIR}")
    PARENT_DIR = "/content"
else:
    # Local: notebook physiographsleep/ içinde, parent'ı sys.path'e ekle
    PROJECT_DIR = os.path.dirname(os.path.abspath("."))
    if os.path.basename(os.getcwd()) != "physiographsleep":
        PROJECT_DIR = os.path.abspath(".")
    PARENT_DIR = os.path.dirname(PROJECT_DIR)
    print(f"Local repo: {PROJECT_DIR}")

if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)
os.chdir(PARENT_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"Project root: {PROJECT_DIR}")


In [ ]:
# Bağımlılıklar
!pip install -q -r {PROJECT_DIR}/requirements.txt
if WANDB_ENABLED:
    !pip install -q wandb


In [ ]:
import torch
import numpy as np
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")


## 2. Configuration

> Tüm yollar `PROJECT_DIR` (= `physiographsleep/` klasörü) altında. Hiçbir şey bu klasörün dışına yazılmaz.

In [ ]:
import os
import sys
from pathlib import Path

# === PROJECT_DIR (Colab vs local) ===
# Repo yapısı:
#   <parent>/physiographsleep/          ← git repo root + Python package
#            ├── configs/ models/ training/ ...
#            ├── main.ipynb  (bu notebook)
#            └── __init__.py
# Colab'da: /content/physiographsleep   (git clone ile oluşur)
# Local'de: cwd zaten neurographTdraft/, physiographsleep/ alt klasör.

if "google.colab" in sys.modules:
    PROJECT_DIR = Path("/content/physiographsleep")
    IS_COLAB = True
else:
    cwd = Path.cwd()
    # Local: workspace root (neurographTdraft/) altında physiographsleep/ var mı?
    if (cwd / "physiographsleep" / "__init__.py").exists():
        PROJECT_DIR = cwd / "physiographsleep"
    elif (cwd / "__init__.py").exists() and cwd.name == "physiographsleep":
        PROJECT_DIR = cwd
    else:
        PROJECT_DIR = cwd
    IS_COLAB = False

# Repo root = package root olduğu için __init__.py ile doğrula
assert (PROJECT_DIR / "__init__.py").exists(), (
    f"physiographsleep package not found at {PROJECT_DIR}\n"
    f"Expected __init__.py in {PROJECT_DIR}"
)
PARENT_DIR = str(PROJECT_DIR.parent)
if PARENT_DIR not in sys.path:
    sys.path.insert(0, PARENT_DIR)
print(f"PROJECT_DIR = {PROJECT_DIR}  (Colab={IS_COLAB})")

# === imports ===
from physiographsleep.configs import ExperimentConfig
from physiographsleep.utils.reproducibility import get_device, set_seed

# === Channel mode (1ch EEG vs 2ch EEG+EOG) ===
USE_EOG = False  # True = 2 kanal (EEG Fpz-Cz + EOG horizontal); False = 1 kanal

# === WandB toggles ===
WANDB_ENABLED = True
WANDB_PROJECT = "neurographTdraft"
WANDB_RUN_NAME = "physiographsleep_sota_revert"

# === Output paths (PROJECT_DIR İÇİNDE — Drive değil) ===
OUTPUTS_DIR = PROJECT_DIR / "outputs"
OUTPUTS_DIR.mkdir(exist_ok=True)

# === Build config ===
config = ExperimentConfig()
# Colab'da /content/data/sleepedf20 (kalıcı olmaz ama hızlı); local'de proje altı.
if IS_COLAB:
    config.data.data_dir = "/content/data/sleepedf20"
    config.data.cache_dir = "/content/cache"
else:
    config.data.data_dir = str(PROJECT_DIR / "data" / "sleepedf20")
    config.data.cache_dir = str(PROJECT_DIR / "cache")
config.train.checkpoint_dir = str(PROJECT_DIR / "checkpoints")
config.train.log_dir = str(PROJECT_DIR / "logs")

# Channel mode patch
if USE_EOG:
    config.data.use_eog = True
    config.model.waveform.in_channels = config.data.num_input_channels  # 2

# Eğitim hiperparametreleri (defaults zaten dengeli; aşağıdaki override'lar
# Sleep-EDF-20 için tarihi kalibrasyon).
# Önceki başarılı run (commit 9511d6d): raw MF1=0.796 / biased=0.808 / HMM=0.813.
config.train.epochs = 60
config.train.lr = 1e-3
config.train.n1_boost = 2.0
# patience varsayılan 10 (config tarafında ayarlandı).

# Pathway / λ-fusion / N1-Mixup ablation toggle'ları BİR SONRAKİ HÜCREDE
# yönetilir. Varsayılan (`True`) → yeni scGraPhT defaults; `False` →
# SOTA-benzeri referans konfigürasyon (commit 9511d6d).

# === Performans toggles ===
USE_TORCH_COMPILE = False

set_seed(config.seed)
device = get_device("auto")


In [ ]:
# ============================================================
# Ablation / SOTA-revert toggles
# ------------------------------------------------------------
# Git arkeolojisi (commit 9511d6d, Apr 18): raw MF1=0.796 / biased=0.808
# / HMM=0.813 sonuçlarını veren referans konfigürasyonda ŞU ÜÇ FEATURE
# KAPALIYDI. Sonraki commit (e7bf922) hepsini aynı anda default AÇIK
# yaptı + wd/drop_path/patience'ı da değiştirdi → regresyon.
#
# Tek değişken A/B yapabilmek için aşağıdaki üç anahtarı bağımsız
# açıp kapatabilirsin:
#   False → feature kapalı (SOTA-benzeri)
#   True  → feature açık (yeni default)
#
# Ablation protokolü: bu hücredeki 3 toggle'ı tek tek değiştirip
# notebook'u baştan sona çalıştır. Her run otomatik olarak
# outputs/ablation/<kombo>.json dosyasına kaydedilir (hücre #VSC-9477da48
# sonunda). En sonda aggregate hücresi hepsini tablolar.
# ============================================================
USE_PATHWAY  = False   # hetero → homo → all-edges pipeline
USE_FUSION   = False   # λ-fusion (auxiliary transformer head)
USE_N1_MIXUP = False   # N1-targeted Mixup augmentation

if not USE_PATHWAY:
    config.model.graph.edge_pathways = None
if not USE_FUSION:
    config.model.fusion = None
if not USE_N1_MIXUP:
    config.train.n1_mixup = None

# Ablation ismi (dosya adı + WandB run name için)
ABLATION_NAME = (
    f"p{int(USE_PATHWAY)}"
    f"_f{int(USE_FUSION)}"
    f"_m{int(USE_N1_MIXUP)}"
    f"_n1boost{config.train.n1_boost:g}"
    f"_wd{config.train.optimizer.weight_decay:g}"
    f"_ls{config.train.loss.label_smoothing:g}"
)

# ============================================================
# Final özet — ablation sonrası efektif konfigürasyon
# ============================================================
print("=" * 60)
print("PATHS")
print("=" * 60)
print(f"  data_dir:       {config.data.data_dir}")
print(f"  cache_dir:      {config.data.cache_dir}")
print(f"  checkpoint_dir: {config.train.checkpoint_dir}")
print(f"  log_dir:        {config.train.log_dir}")
print(f"  outputs_dir:    {OUTPUTS_DIR}")
print("=" * 60)
print(f"Device:           {device}")
print(f"Subjects:         {config.data.num_subjects} (train={config.data.train_subjects}, val={config.data.val_subjects})")
print(f"Batch size:       {config.data.batch_size}")
print(f"Num workers:      {config.data.num_workers}")
print(f"Wake-trim (min):  {config.data.wake_trim_minutes}")
print(f"Channel mode:     {'EEG Fpz-Cz + EOG horizontal (2ch)' if USE_EOG else 'EEG Fpz-Cz only (1ch)'}")
print(f"WaveformStem in:  {config.model.waveform.in_channels} channel(s)")
print(f"torch.compile:    {USE_TORCH_COMPILE}")
print(f"Epochs:           {config.train.epochs}  (early stop patience={config.train.patience})")
print(f"LR / N1 boost:    {config.train.lr} / {config.train.n1_boost}x")
print(f"LR warmup:        {config.train.scheduler.warmup_epochs} epochs  (linear 1%→100%)")
print(f"Graph layers/heads: {config.model.graph.num_layers} / {config.model.graph.num_heads}  (DropPath={config.model.graph.drop_path})")
print("=" * 60)
print("ABLATION TOGGLES")
print("=" * 60)
print(f"  ablation_name:    {ABLATION_NAME}")
print(f"  pathway:          {'ON ' if USE_PATHWAY  else 'OFF'}  → edge_pathways={config.model.graph.edge_pathways}")
fusion_str = (
    f"init_lambda={config.model.fusion.init_lambda} detach_gnn={config.model.fusion.detach_gnn_for_lambda}"
    if config.model.fusion is not None else "DISABLED"
)
print(f"  lambda-fusion:    {'ON ' if USE_FUSION   else 'OFF'}  → {fusion_str}")
mix_str = (
    f"prob={config.train.n1_mixup.prob} alpha={config.train.n1_mixup.alpha}"
    if config.train.n1_mixup is not None else "DISABLED"
)
print(f"  N1-Mixup:         {'ON ' if USE_N1_MIXUP else 'OFF'}  → {mix_str}")
print(f"Label smoothing:  {config.train.loss.label_smoothing}  | WD: {config.train.optimizer.weight_decay}")


In [ ]:
# WandB init (opsiyonel)
wandb_run = True
if WANDB_ENABLED:
    import wandb
    # Colab: secrets'tan API key alın veya kullanıcı interaktif login olur
    try:
        from google.colab import userdata  # type: ignore
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    except Exception:
        pass  # Local: WANDB_API_KEY env veya wandb login

    wandb_run = wandb.init(
        project=WANDB_PROJECT,
        name=WANDB_RUN_NAME,
        dir=OUTPUTS_DIR,
        config={
            "num_subjects": config.data.num_subjects,
            "batch_size": config.data.batch_size,
            "seq_len": config.data.seq_len,
            "epochs": config.train.epochs,
            "lr": config.train.lr,
            "n1_boost": config.train.n1_boost,
            "patience": config.train.patience,
        },
    )
    print(f"WandB run: {wandb_run.url}")
else:
    print("WandB devre dışı (WANDB_ENABLED=False)")


## 3. Veri Yükleme + Sınıf Dağılımı

In [ ]:
from physiographsleep.data.loader import load_sleep_edf

data = load_sleep_edf(config.data)

STAGE_NAMES = ["W", "N1", "N2", "N3", "REM"]
print("Split sizes:")
for split_name, split_data in data.items():
    epochs = split_data["epochs"]
    labels = split_data["labels"]
    spectral = split_data.get("spectral")
    print(f"  {split_name:5s}: {len(labels):6d} epochs | shape={epochs.shape} | spectral={spectral.shape if spectral is not None else 'N/A'}")

print("\nTrain class distribution:")
train_labels = data["train"]["labels"]
counts = np.bincount(train_labels, minlength=5)
total = len(train_labels)
for i, name in enumerate(STAGE_NAMES):
    pct = 100 * counts[i] / total
    bar = "█" * int(pct / 2)
    print(f"  {name:3s}: {counts[i]:6d} ({pct:5.1f}%) {bar}")


## 4. DataLoaders

In [ ]:
from torch.utils.data import DataLoader
from physiographsleep.data.dataset import SleepEDFDataset
from physiographsleep.data.transforms import SleepTransforms

train_transform = SleepTransforms(config.data)

train_ds = SleepEDFDataset(
    data["train"]["epochs"], data["train"]["labels"],
    config=config.data, transform=train_transform,
    spectral=data["train"].get("spectral"),
)
val_ds = SleepEDFDataset(
    data["val"]["epochs"], data["val"]["labels"],
    config=config.data,
    spectral=data["val"].get("spectral"),
)

val_loader = DataLoader(
    val_ds, batch_size=config.data.batch_size,
    shuffle=False, num_workers=config.data.num_workers,
    pin_memory=True,
    persistent_workers=config.data.num_workers > 0,
    prefetch_factor=4 if config.data.num_workers > 0 else None,
)

print(f"Train: {len(train_ds)} samples")
print(f"Val:   {len(val_ds)} samples, {len(val_loader)} batches")


## 5. Model + Sanity Check

In [ ]:
from physiographsleep.models.physiographsleep import PhysioGraphSleep

model = PhysioGraphSleep(config.model)
param_counts = model.count_parameters()

print("Parameter counts:")
for name, count in param_counts.items():
    print(f"  {name:20s}: {count:>10,}")

# Sanity forward
batch = next(iter(val_loader))
with torch.no_grad():
    model.eval()
    sig = batch["signal"][:2]
    spec = batch["spectral"][:2] if "spectral" in batch else torch.zeros(2, config.data.seq_len, 5, 42)
    out = model(sig, spec)
print(f"\nSanity forward: signal={tuple(sig.shape)} → stage={tuple(out['stage'].shape)} OK")

# torch.compile (opsiyonel — ilk epoch +60s, sonraki epoch'lar 1.3-2x hızlı)
if USE_TORCH_COMPILE and torch.cuda.is_available():
    print("\ntorch.compile aktif (ilk derleme uzun sürebilir)...")
    model = torch.compile(model, mode="reduce-overhead", dynamic=False)


## 6. Training — Live Plots + WandB + Detailed Epoch Table

Bu hücre `Trainer.callback` hook'unu kullanarak her epoch sonunda:
- Detaylı tablo satırı yazdırır (train loss/acc/MF1, val acc/MF1/κ, per-class F1)
- 4 panelli live plot çizer (loss, accuracy, macro-F1, per-class val F1)
- WandB'ye metrikleri loglar (etkinse)

In [ ]:
import gc
import json
import matplotlib.pyplot as plt
from IPython.display import clear_output, display

from physiographsleep.models.losses import MultiTaskLoss
from physiographsleep.data.spectral import SpectralFeatureExtractor
from physiographsleep.training.trainer import Trainer
from physiographsleep.utils.logging_utils import setup_logger

# Free GPU memory
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated, {torch.cuda.memory_reserved()/1e9:.2f} GB reserved")

# NOTE: class_weights'i KASITLI OLARAK kaldırdık.
# Sampler zaten N1'i boost'luyor (WeightedRandomSampler with n1_boost).
# Aynı anda inverse-frequency class_weights uygulamak N1'i
# efektif over-weight ediyor ve F1_N1'yi platoya sokuyor.
# Tek mekanizma: sampler-level N1 boost. Loss uniform kalır.
loss_fn = MultiTaskLoss(config.train.loss)
spectral_ext = SpectralFeatureExtractor(config.data)
logger = setup_logger(log_dir=config.train.log_dir)
print(f"Loss: FocalLoss(gamma={config.train.loss.focal_gamma}, label_smoothing={config.train.loss.label_smoothing}) — class_weights disabled (sampler handles N1)")


In [ ]:
# ============================================================
# Training history + per-epoch callback
# ============================================================
HISTORY = {"epoch": [],
           "train_loss": [], "val_loss": [],
           "train_acc": [], "train_mf1": [],
           "val_acc": [], "val_mf1": [], "val_mf1_gnn": [], "val_kappa": [],
           "val_f1_W": [], "val_f1_N1": [], "val_f1_N2": [],
           "val_f1_N3": [], "val_f1_REM": []}

HEADER_PRINTED = {"v": False}
def _print_header():
    if HEADER_PRINTED["v"]:
        return
    cols = ("Epoch", "TrLoss", "VlLoss",
            "TrAcc", "TrMF1", "VlAcc", "VlMF1", "GnnMF1", "Kappa",
            "F1_W", "F1_N1", "F1_N2", "F1_N3", "F1_REM")
    print(" | ".join(f"{c:>7s}" for c in cols))
    print("-" * (10 * len(cols)))
    HEADER_PRINTED["v"] = True

def _draw_live_plot():
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    epochs_x = list(range(len(HISTORY["epoch"])))

    axes[0, 0].plot(epochs_x, HISTORY["train_loss"], "-o", ms=3, label="train")
    axes[0, 0].plot(epochs_x, HISTORY["val_loss"],   "-s", ms=3, label="val")
    axes[0, 0].set_title("Loss"); axes[0, 0].set_xlabel("epoch"); axes[0, 0].grid(alpha=0.3)
    axes[0, 0].legend()

    axes[0, 1].plot(epochs_x, HISTORY["train_acc"], "-o", ms=3, label="train")
    axes[0, 1].plot(epochs_x, HISTORY["val_acc"],   "-s", ms=3, label="val")
    axes[0, 1].set_title("Accuracy"); axes[0, 1].set_ylim(0, 1); axes[0, 1].grid(alpha=0.3)
    axes[0, 1].legend()

    axes[1, 0].plot(epochs_x, HISTORY["train_mf1"], "-o", ms=3, label="train")
    axes[1, 0].plot(epochs_x, HISTORY["val_mf1"],   "-s", ms=3, label="val (fused)")
    # GNN-only MF1 — fusion kapalıyken val_mf1 ile aynı; açıkken ayrık eğri
    if any(v is not None for v in HISTORY["val_mf1_gnn"]):
        gnn_y = [v if v is not None else float("nan") for v in HISTORY["val_mf1_gnn"]]
        axes[1, 0].plot(epochs_x, gnn_y, "-^", ms=3, label="val (gnn-only)")
    axes[1, 0].set_title("Macro-F1"); axes[1, 0].set_ylim(0, 1); axes[1, 0].grid(alpha=0.3)
    axes[1, 0].legend()

    for cls in ["W", "N1", "N2", "N3", "REM"]:
        axes[1, 1].plot(epochs_x, HISTORY[f"val_f1_{cls}"], "-o", ms=3, label=cls)
    axes[1, 1].set_title("Val per-class F1"); axes[1, 1].set_ylim(0, 1); axes[1, 1].grid(alpha=0.3)
    axes[1, 1].legend(ncol=5, fontsize=8)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUTS_DIR, "training_curves.png"), dpi=80, bbox_inches="tight")
    clear_output(wait=True)
    display(fig)
    plt.close(fig)


def epoch_callback(epoch, train_loss, val_loss, train_metrics, val_metrics):
    HISTORY["epoch"].append(epoch)
    HISTORY["train_loss"].append(train_loss)
    HISTORY["val_loss"].append(val_loss)
    HISTORY["train_acc"].append(train_metrics.get("accuracy", 0.0))
    HISTORY["train_mf1"].append(train_metrics.get("macro_f1", 0.0))
    HISTORY["val_acc"].append(val_metrics.get("accuracy", 0.0))
    HISTORY["val_mf1"].append(val_metrics.get("macro_f1", 0.0))
    HISTORY["val_mf1_gnn"].append(val_metrics.get("macro_f1_gnn"))  # None ise fusion kapalı
    HISTORY["val_kappa"].append(val_metrics.get("kappa", 0.0))
    for cls in ["W", "N1", "N2", "N3", "REM"]:
        HISTORY[f"val_f1_{cls}"].append(val_metrics.get(f"f1_{cls}", 0.0))

    _print_header()
    gnn_mf1 = val_metrics.get("macro_f1_gnn")
    gnn_str = f"{gnn_mf1:>7.4f}" if gnn_mf1 is not None else "   N/A "
    row = (
        f"{epoch:>7d}",
        f"{train_loss:>7.4f}", f"{val_loss:>7.4f}",
        f"{train_metrics.get('accuracy', 0):>7.4f}",
        f"{train_metrics.get('macro_f1', 0):>7.4f}",
        f"{val_metrics.get('accuracy', 0):>7.4f}",
        f"{val_metrics.get('macro_f1', 0):>7.4f}",
        gnn_str,
        f"{val_metrics.get('kappa', 0):>7.4f}",
        f"{val_metrics.get('f1_W', 0):>7.4f}",
        f"{val_metrics.get('f1_N1', 0):>7.4f}",
        f"{val_metrics.get('f1_N2', 0):>7.4f}",
        f"{val_metrics.get('f1_N3', 0):>7.4f}",
        f"{val_metrics.get('f1_REM', 0):>7.4f}",
    )
    print(" | ".join(row))

    if wandb_run is not None:
        log = {
            "train/loss": train_loss,
            "val/loss": val_loss,
            "train/acc": train_metrics.get("accuracy", 0),
            "train/macro_f1": train_metrics.get("macro_f1", 0),
            "val/acc": val_metrics.get("accuracy", 0),
            "val/macro_f1": val_metrics.get("macro_f1", 0),
            "val/kappa": val_metrics.get("kappa", 0),
        }
        for cls in ["W", "N1", "N2", "N3", "REM"]:
            log[f"val/f1_{cls}"] = val_metrics.get(f"f1_{cls}", 0)
        # λ-fusion açıkken pure-GNN baseline'ı da logla (diagnostic)
        if gnn_mf1 is not None:
            log["val/macro_f1_gnn"] = gnn_mf1
            log["val/acc_gnn"] = val_metrics.get("accuracy_gnn", 0)
        # step=epoch + commit=True → canlı web dashboard güncellenir
        wandb_run.log(log, step=epoch, commit=True)

    _draw_live_plot()

    with open(os.path.join(OUTPUTS_DIR, "history.json"), "w") as f:
        json.dump(HISTORY, f, indent=2)


trainer = Trainer(
    model=model,
    loss_fn=loss_fn,
    train_dataset=train_ds,
    train_labels=data["train"]["labels"],
    val_loader=val_loader,
    config=config.train,
    data_config=config.data,
    device=device,
    spectral_extractor=spectral_ext,
    callback=epoch_callback,
)
print("Trainer hazır. Sonraki hücre eğitimi başlatır.")


In [ ]:
# Auto-resume + run training
from physiographsleep.utils.io_utils import load_checkpoint
from physiographsleep.evaluation.metrics import MetricsCalculator

ckpt_path = os.path.join(config.train.checkpoint_dir, config.train.checkpoint_name)
if os.path.exists(ckpt_path):
    ckpt = load_checkpoint(ckpt_path, device)
    model.load_state_dict(ckpt["model"])
    prev_mf1 = ckpt["metrics"]["macro_f1"]
    print(f"Mevcut checkpoint bulundu: {ckpt_path} (val MF1={prev_mf1:.4f})")
    print("→ Yeniden eğitmek istemiyorsan bu hücreyi atla, post-processing'e geç.")
    print("→ Yeniden eğitmek için checkpoint dosyasını silip bu hücreyi tekrar çalıştır.")

best_metrics = trainer.train()
print("\n" + "=" * 50)
print("TRAINING COMPLETE")
print("=" * 50)
print(MetricsCalculator.format_report(best_metrics))


## 7. Test Eval + HMM Post-processing + Confusion Matrices

In [ ]:
import torch.nn.functional as F

from physiographsleep.training.evaluator import Evaluator
from physiographsleep.evaluation.metrics import MetricsCalculator
from physiographsleep.evaluation.postprocessing import HMMPostProcessor, LogitBiasOptimizer
from physiographsleep.evaluation.visualization import plot_confusion_matrix

# En iyi checkpoint'i yükle
ckpt_path = os.path.join(config.train.checkpoint_dir, config.train.checkpoint_name)
if os.path.exists(ckpt_path):
    ckpt = load_checkpoint(ckpt_path, device)
    model.load_state_dict(ckpt["model"])
    print(f"Yüklendi: {ckpt_path} (val MF1={ckpt['metrics']['macro_f1']:.4f})")
else:
    print("Checkpoint bulunamadı, mevcut model state kullanılıyor")

evaluator = Evaluator(device)

if "test" in data:
    test_ds = SleepEDFDataset(
        data["test"]["epochs"], data["test"]["labels"],
        config=config.data,
        spectral=data["test"].get("spectral"),
    )
    test_loader = DataLoader(
        test_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )

    # Test ve val logits
    test_metrics, test_logits, test_labels_arr = evaluator.evaluate(
        model, test_loader, spectral_ext, return_logits=True,
    )
    val_loader_eval = DataLoader(
        val_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers,
    )
    _, val_logits, val_labels_arr = evaluator.evaluate(
        model, val_loader_eval, spectral_ext, return_logits=True,
    )

    calc = MetricsCalculator()
    results = []  # [(name, predictions, metrics), ...]

    # [1] Raw argmax — baseline
    preds_raw = test_logits.argmax(axis=1)
    raw_m = calc.compute_all(test_labels_arr, preds_raw)
    results.append(("raw", preds_raw, raw_m))

    # [2] Logit-bias (val set üzerinde Nelder-Mead ile macro-F1 maksimize):
    # AttnSleep / SleepTransformer'ın class-imbalance düzeltmesi.
    bias_opt = LogitBiasOptimizer(num_classes=5)
    bias_opt.fit(val_logits, val_labels_arr)
    preds_biased = bias_opt.apply(test_logits)
    biased_m = calc.compute_all(test_labels_arr, preds_biased)
    results.append(("biased", preds_biased, biased_m))

    # [3] HMM posterior Viterbi smoothing (DeepSleepNet/AttnSleep standart):
    # Transition matrix train labels'tan, log-softmax(bias-adjusted logits)
    # üzerinden Viterbi decoding.
    adjusted_logits = test_logits + bias_opt.bias
    log_post = F.log_softmax(torch.from_numpy(adjusted_logits), dim=-1).numpy()
    hmm = HMMPostProcessor(num_classes=5).fit(data["train"]["labels"])
    preds_hmm = hmm.smooth_posteriors(log_post)
    hmm_m = calc.compute_all(test_labels_arr, preds_hmm)
    results.append(("hmm", preds_hmm, hmm_m))

    print("\n" + "=" * 50)
    print("TEST RESULTS")
    print("=" * 50)
    prev = None
    for name, _, m in results:
        delta = f"  ({m['macro_f1']-prev['macro_f1']:+.4f})" if prev else ""
        print(f"\n[{name}] MF1={m['macro_f1']:.4f}{delta}")
        print(MetricsCalculator.format_report(m))
        prev = m

    # Confusion matrices → outputs/
    for name, preds, m in results:
        cm = MetricsCalculator.confusion_matrix(test_labels_arr, preds)
        plot_confusion_matrix(
            cm,
            save_path=os.path.join(OUTPUTS_DIR, f"cm_{name}.png"),
            title=f"{name} (MF1={m['macro_f1']:.3f})",
        )
    print(f"\nConfusion matrices kaydedildi: {OUTPUTS_DIR}")

    # Final dump
    final_metrics = {name: m for name, _, m in results}
    final_path = os.path.join(OUTPUTS_DIR, "final_model.pt")
    torch.save({
        "model": model.state_dict(),
        "config": {"num_subjects": config.data.num_subjects, "batch_size": config.data.batch_size,
                   "seq_len": config.data.seq_len},
        "metrics": final_metrics,
        "param_counts": param_counts,
    }, final_path)
    print(f"Final model: {final_path}  ({param_counts['total']:,} params)")

    if wandb_run is not None:
        for name, _, m in results:
            wandb_run.summary[f"test_{name}_mf1"] = m["macro_f1"]
            wandb_run.summary[f"test_{name}_acc"] = m["accuracy"]
            for cls in ["W", "N1", "N2", "N3", "REM"]:
                wandb_run.summary[f"test_{name}_f1_{cls}"] = m.get(f"f1_{cls}", 0)
        wandb_run.save(os.path.join(OUTPUTS_DIR, "*.png"))
        wandb_run.save(os.path.join(OUTPUTS_DIR, "history.json"))
        wandb_run.finish()
        print("WandB run kapatıldı")
else:
    print("Test split yok")


In [ ]:
# ============================================================
# 7.1 — Ablation result logger
# ------------------------------------------------------------
# Her run'ın sonucunu outputs/ablation/<ABLATION_NAME>.json dosyasına
# yazar. Ablation tablosu (en sondaki hücre) bu dosyaları okuyarak
# karşılaştırma üretir. Her toggle kombinasyonu için notebook'u baştan
# sona bir kez çalıştır → her run kendi JSON'unu kaydeder → aggregate
# hücresi hepsini basar.
# ============================================================
import json
from datetime import datetime

if "test" not in data:
    print("Test split yok, ablation log atlandı.")
else:
    ablation_dir = OUTPUTS_DIR / "ablation"
    ablation_dir.mkdir(exist_ok=True)
    out_path = ablation_dir / f"{ABLATION_NAME}.json"

    record = {
        "ablation_name": ABLATION_NAME,
        "timestamp": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        "toggles": {
            "pathway":  USE_PATHWAY,
            "fusion":   USE_FUSION,
            "n1_mixup": USE_N1_MIXUP,
        },
        "hyperparams": {
            "epochs":          config.train.epochs,
            "patience":        config.train.patience,
            "lr":              config.train.lr,
            "weight_decay":    config.train.optimizer.weight_decay,
            "label_smoothing": config.train.loss.label_smoothing,
            "n1_boost":        config.train.n1_boost,
            "warmup_epochs":   config.train.scheduler.warmup_epochs,
            "t_max":           config.train.scheduler.t_max,
            "drop_path":       config.model.graph.drop_path,
        },
        "param_counts": {k: int(v) for k, v in param_counts.items()},
        "best_val_mf1": float(ckpt["metrics"]["macro_f1"]) if os.path.exists(ckpt_path) else None,
        "test": {
            name: {
                "accuracy":    float(m["accuracy"]),
                "macro_f1":    float(m["macro_f1"]),
                "kappa":       float(m.get("kappa", 0.0)),
                "mcc":         float(m.get("mcc", 0.0)),
                "f1_W":        float(m.get("f1_W", 0.0)),
                "f1_N1":       float(m.get("f1_N1", 0.0)),
                "f1_N2":       float(m.get("f1_N2", 0.0)),
                "f1_N3":       float(m.get("f1_N3", 0.0)),
                "f1_REM":      float(m.get("f1_REM", 0.0)),
            }
            for name, _, m in results
        },
    }
    with open(out_path, "w") as f:
        json.dump(record, f, indent=2)
    print(f"\nAblation kaydedildi: {out_path}")
    print(f"  HMM MF1 = {record['test']['hmm']['macro_f1']:.4f} | "
          f"N1 F1 = {record['test']['hmm']['f1_N1']:.4f} | "
          f"κ = {record['test']['hmm']['kappa']:.4f}")


## 8. Component Diagnostics — Her Katmanın Gerçek Katkısı

Bu bölüm, eğitilmiş modeldeki **her katmanın gerçekten işe yarayıp yaramadığını** ölçer.
Tüm hücreler `best.pt` (EMA checkpoint) yüklendikten sonra çalışır — eğitimi tekrar
başlatmaz, sadece forward pass yapar.

**Diagnostic akışı:**

| # | Hücre | Sorusu | Kırmızı bayrak |
|---|-------|--------|----------------|
| 8.1 | Activation stats | Katman ölü mü / patlıyor mu? | `std < 0.01` veya `dead% > 50` veya NaN/Inf |
| 8.2 | Linear probe per stage | Her stage discrimination ekliyor mu? | Bir sonraki stage MF1'i **artırmıyorsa** o katman fazlalık |
| 8.3 | Input modality ablation | Waveform mı spectral mı baskın? | Tek modalite ile MF1 benzerse diğeri zayıf |
| 8.4 | Fusion + λ inspection | Fusion hangi branch'e güveniyor? | σ(λ) ≈ 0 veya ≈ 1 ise fusion aslında single-branch |

Çıktıları bana at, hangi katmanın gelişmesi gerektiğine birlikte karar verelim.


In [ ]:
# ============================================================
# 8.1 — Activation statistics via forward hooks
# ------------------------------------------------------------
# Her ana bloğun çıkışından tek-batch istatistik topla:
#   mean / std / min / max / dead%  + NaN / Inf kontrolü
# Kırmızı bayraklar:
#   - std < 0.01       → temsil çökmüş (dead layer)
#   - dead% > 50       → ReLU sonrası çok fazla ölü nöron
#   - NaN veya Inf     → sayısal patlama
#   - mean |μ| > 3·σ   → normalization eksik
# ============================================================
import torch

_stats_cache = {}

def _tensor_stats(t: torch.Tensor) -> dict:
    t = t.detach().float()
    return {
        "shape": tuple(t.shape),
        "mean":  t.mean().item(),
        "std":   t.std().item(),
        "min":   t.min().item(),
        "max":   t.max().item(),
        "dead%": (t.abs() < 1e-6).float().mean().item() * 100.0,
        "nan":   bool(torch.isnan(t).any().item()),
        "inf":   bool(torch.isinf(t).any().item()),
    }

def _make_hook(name: str):
    def hook(_mod, _inp, out):
        t = out[0] if isinstance(out, tuple) else out
        if name not in _stats_cache and isinstance(t, torch.Tensor):
            _stats_cache[name] = _tensor_stats(t)
    return hook

# Register hooks on major blocks
_handles = []
_targets = ["waveform_stem", "spectral_encoder", "graph_encoder", "sequence_decoder", "heads"]
if model.fusion_enabled:
    _targets += ["transformer_classifier", "fusion"]
for _name in _targets:
    _h = getattr(model, _name).register_forward_hook(_make_hook(_name))
    _handles.append(_h)

# One forward pass over a val batch
model.eval()
with torch.no_grad():
    _batch = next(iter(val_loader))
    _sig  = _batch["signal"].to(device)
    _spec = _batch["spectral"].to(device)
    _mask = _batch.get("mask", None)
    if _mask is not None:
        _mask = _mask.to(device)
    _ = model(_sig, _spec, _mask)

for _h in _handles:
    _h.remove()

# Pretty print
print(f"{'Module':<24s} {'Shape':<26s} {'μ':>9s} {'σ':>9s} {'min':>9s} {'max':>9s} {'dead%':>7s}")
print("-" * 96)
for _name, _s in _stats_cache.items():
    flag = ""
    if _s["nan"] or _s["inf"]:
        flag = "  <-- NaN/Inf"
    elif _s["std"] < 0.01:
        flag = "  <-- COLLAPSED (std<0.01)"
    elif _s["dead%"] > 50:
        flag = f"  <-- DEAD ({_s['dead%']:.0f}% near-zero)"
    print(
        f"{_name:<24s} {str(_s['shape']):<26s} "
        f"{_s['mean']:>9.3f} {_s['std']:>9.3f} {_s['min']:>9.3f} {_s['max']:>9.3f} "
        f"{_s['dead%']:>6.1f}%{flag}"
    )


In [ ]:
# ============================================================
# 8.2 — Linear probe per stage
# ------------------------------------------------------------
# Her stage'in ÇIKTI feature'ını dondurup üzerine LogisticRegression
# eğitiyoruz. Eğer bir sonraki stage MF1'i artırmıyorsa → o katman
# discriminatif bir şey eklememiş demektir (fazlalık / yeniden tasarlanabilir).
#
# Aşamalar:
#   raw_concat : waveform patches + spectral bands (stem+spectral çıkışı)
#   graph      : HeteroGraphEncoder epoch embedding (128-d)
#   sequence   : SequenceTransitionDecoder center output (160-d)
# ============================================================
if "test_loader" not in globals():
    print("test_loader tanımsız (test split yok). Bu diagnostic atlandı.")
else:
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import f1_score, accuracy_score

    _probe_val_loader = DataLoader(
        val_ds, batch_size=config.data.batch_size,
        shuffle=False, num_workers=config.data.num_workers, pin_memory=True,
    )

    def _extract_stage_features(model, loader):
        feats = {"raw_concat": [], "graph": [], "sequence": []}
        labels = []
        model.eval()
        with torch.no_grad():
            for batch in loader:
                sig  = batch["signal"].to(device)
                spec = batch["spectral"].to(device)
                mask = batch.get("mask", None)
                if mask is not None:
                    mask = mask.to(device)
                B, L, C, T = sig.shape
                center = L // 2

                # Flatten sequence → (B*L, …)
                flat_sig  = sig.reshape(B * L, C, T)
                flat_spec = spec.reshape(B * L, *spec.shape[2:])

                patch = model.waveform_stem(flat_sig)        # (B*L, P, D)
                band  = model.spectral_encoder(flat_spec)    # (B*L, K, D)
                epoch_emb_flat = model.graph_encoder(patch, band)  # (B*L, d_g)
                epoch_emb = epoch_emb_flat.reshape(B, L, -1)

                # raw_concat (center epoch only): patch + band tokens flattened
                P_c = patch.reshape(B, L, -1)[:, center]     # (B, P*D)
                K_c = band.reshape(B, L, -1)[:, center]      # (B, K*D)
                raw_concat = torch.cat([P_c, K_c], dim=-1)   # (B, P*D + K*D)

                graph_c = epoch_emb[:, center]               # (B, d_g)

                seq_out = model.sequence_decoder(epoch_emb, mask)  # (B, L, d_s)
                seq_c = seq_out[:, center]                   # (B, d_s)

                feats["raw_concat"].append(raw_concat.cpu().numpy())
                feats["graph"].append(graph_c.cpu().numpy())
                feats["sequence"].append(seq_c.cpu().numpy())
                labels.append(batch["label"].numpy())

        feats = {k: np.concatenate(v, axis=0) for k, v in feats.items()}
        return feats, np.concatenate(labels)

    print("Extracting features (val → probe train, test → probe eval) ...")
    val_feats,  val_y  = _extract_stage_features(model, _probe_val_loader)
    test_feats, test_y = _extract_stage_features(model, test_loader)

    print(f"\n{'Stage':<14s} {'Dim':>8s} {'Probe MF1':>11s} {'Probe Acc':>11s} {'ΔMF1 vs prev':>14s}")
    print("-" * 62)
    prev_mf1 = None
    for stage in ["raw_concat", "graph", "sequence"]:
        Xtr = val_feats[stage]
        Xte = test_feats[stage]
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(
            max_iter=400, n_jobs=-1, class_weight="balanced", solver="lbfgs",
        ).fit(sc.transform(Xtr), val_y)
        preds = clf.predict(sc.transform(Xte))
        mf1 = f1_score(test_y, preds, average="macro")
        acc = accuracy_score(test_y, preds)
        delta = f"{mf1 - prev_mf1:+.4f}" if prev_mf1 is not None else "   base"
        print(f"{stage:<14s} {Xtr.shape[1]:>8d} {mf1:>11.4f} {acc:>11.4f} {delta:>14s}")
        prev_mf1 = mf1

    print("\nYorum:")
    print("  raw_concat → graph:  GNN bir şey ekliyor mu?")
    print("  graph → sequence  :  sequence decoder gerçekten context kullanıyor mu?")
    print("  Δ negatif veya ~0 → o katman discrimination kazandırmıyor (yeniden tasarlanmalı).")


In [ ]:
# ============================================================
# 8.3 — Input modality ablation
# ------------------------------------------------------------
# Test setinde spectral VEYA waveform'u sıfırla, tekrar değerlendir.
# Eğer:
#   - Sadece waveform varken MF1 ≈ baseline → spectral branch fazlalık
#   - Sadece spectral varken MF1 ≈ baseline → waveform branch fazlalık
#   - İkisi birden gerekli → ideal füzyon durumu (her iki branch katkı sağlıyor)
#
# Not: Sadece INPUT sıfırlanır, model parametreleri sabit.
# UYARI: waveform_stem'de Conv1d(bias=False) + GroupNorm kullanıldığından
# "zero signal" → neredeyse tam sıfır çıktı; spectral branch'te Linear(bias=True)
# olduğundan "zero spectral" → non-zero bias output. Yani asimetrik bir drop
# beklenir; yorumlarken bunu unutma.
# ============================================================
if "test_loader" not in globals():
    print("test_loader tanımsız (test split yok). Bu diagnostic atlandı.")
else:
    import numpy as np
    from sklearn.metrics import f1_score, accuracy_score

    def _eval_with_modality(model, zero_spectral: bool = False, zero_waveform: bool = False):
        all_preds, all_labels = [], []
        model.eval()
        with torch.no_grad():
            for batch in test_loader:
                sig  = batch["signal"].to(device)
                spec = batch["spectral"].to(device)
                if zero_spectral:
                    spec = torch.zeros_like(spec)
                if zero_waveform:
                    sig = torch.zeros_like(sig)
                mask = batch.get("mask", None)
                if mask is not None:
                    mask = mask.to(device)
                out = model(sig, spec, mask)
                all_preds.append(out["stage"].argmax(dim=-1).cpu().numpy())
                all_labels.append(batch["label"].numpy())
        preds  = np.concatenate(all_preds)
        labels = np.concatenate(all_labels)
        return (
            f1_score(labels, preds, average="macro"),
            accuracy_score(labels, preds),
        )

    print(f"{'Condition':<32s} {'MF1':>8s} {'ΔMF1':>9s} {'Acc':>8s}")
    print("-" * 60)
    base_mf1, base_acc = _eval_with_modality(model)
    print(f"{'Baseline (both inputs)':<32s} {base_mf1:>8.4f} {'base':>9s} {base_acc:>8.4f}")

    m, a = _eval_with_modality(model, zero_spectral=True)
    print(f"{'Only waveform (spec=0)':<32s} {m:>8.4f} {m - base_mf1:>+9.4f} {a:>8.4f}")

    m, a = _eval_with_modality(model, zero_waveform=True)
    print(f"{'Only spectral (signal=0)':<32s} {m:>8.4f} {m - base_mf1:>+9.4f} {a:>8.4f}")

    m, a = _eval_with_modality(model, zero_spectral=True, zero_waveform=True)
    print(f"{'Both zero (sanity check)':<32s} {m:>8.4f} {m - base_mf1:>+9.4f} {a:>8.4f}")
    print("\nYorum:")
    print("  Her iki modalite ~eşit drop → dengeli füzyon.")
    print("  Bir branch'in drop'u büyük (örn. -0.15), diğeri küçük (-0.02) →")
    print("    küçük olan branch zayıf, geliştirme odağı bu.")
    print("  Her iki tek-modalite ~baseline → model ikisine de bağımlı değil, aux loss'tan öğreniyor.")


In [ ]:
# ============================================================
# 8.4 — Fusion diagnostics (λ + GNN vs Transformer branch)
# ------------------------------------------------------------
# Sadece USE_FUSION=True ise anlamlı. Raporlar:
#   - σ(λ)  : son λ değeri (transformer branch ağırlığı)
#   - GNN-only MF1    : sadece gnn_logits kullanıldığında
#   - Trans-only MF1  : sadece transformer_logits kullanıldığında
#   - Fused MF1       : asıl çıktı (stage)
#
# λ ≈ 0 → fusion aslında pure-GNN (transformer branch kullanılmamış).
# λ ≈ 1 → fusion aslında pure-transformer (GNN branch ihmal edilmiş).
# 0.2–0.8 arası → gerçek ensemble davranışı.
# ============================================================
if "test_loader" not in globals():
    print("test_loader tanımsız (test split yok). Bu diagnostic atlandı.")
elif not getattr(model, "fusion_enabled", False):
    print("Fusion disabled (USE_FUSION=False). Bu hücreyi atlayabilirsin.")
else:
    import numpy as np
    from sklearn.metrics import f1_score, accuracy_score

    # Final λ
    with torch.no_grad():
        lam = model.fusion.lambda_value.item()
    print(f"sigmoid(λ) = {lam:.4f}")
    print(f"  transformer branch: {lam*100:.1f}%")
    print(f"  GNN branch        : {(1-lam)*100:.1f}%")
    if lam < 0.05 or lam > 0.95:
        print("  UYARI: fusion dejenere durumda (tek branch baskın).")

    # Branch-wise MF1 on test set
    preds_fused, preds_gnn, preds_trans, labels = [], [], [], []
    model.eval()
    with torch.no_grad():
        for batch in test_loader:
            sig  = batch["signal"].to(device)
            spec = batch["spectral"].to(device)
            mask = batch.get("mask", None)
            if mask is not None:
                mask = mask.to(device)
            out = model(sig, spec, mask)
            preds_fused.append(out["stage"].argmax(dim=-1).cpu().numpy())
            preds_gnn.append(out["stage_gnn"].argmax(dim=-1).cpu().numpy())
            preds_trans.append(out["stage_transformer"].argmax(dim=-1).cpu().numpy())
            labels.append(batch["label"].numpy())

    labels = np.concatenate(labels)
    rows = [
        ("fused (output)", np.concatenate(preds_fused)),
        ("gnn-only",       np.concatenate(preds_gnn)),
        ("trans-only",     np.concatenate(preds_trans)),
    ]
    print(f"\n{'Branch':<18s} {'MF1':>8s} {'Acc':>8s} {'F1_N1':>8s}")
    print("-" * 46)
    for name, p in rows:
        mf1 = f1_score(labels, p, average="macro")
        acc = accuracy_score(labels, p)
        f1n1 = f1_score(labels, p, average=None, labels=[0, 1, 2, 3, 4])[1]
        print(f"{name:<18s} {mf1:>8.4f} {acc:>8.4f} {f1n1:>8.4f}")

    print("\nYorum:")
    print("  fused ≥ max(gnn-only, trans-only) → fusion işe yarıyor.")
    print("  fused ≈ gnn-only ise transformer branch efektif olarak ölü → λ→0.")
    print("  fused < max(branch) → fusion zararlı (λ yanlış öğrenmiş, devre dışı bırak).")


### Sonuçları paylaşırken nelere bakmalı?

Aşağıdaki çıktıları olduğu gibi bana at:

1. **8.1 Activation stats tablosu** — sadece `Module | Shape | μ | σ | min | max | dead%` satırları yeterli.
2. **8.2 Linear probe MF1'leri** — `raw_concat` → `graph` → `sequence` geçişindeki ΔMF1.
3. **8.3 Modality ablation** — 4 satırlık tablo (baseline, only waveform, only spectral, both zero).
4. **8.4 Fusion tablosu** (varsa) — σ(λ) + fused/gnn-only/trans-only MF1.

Bu 4 tablodan **her katmanın ağırlığını ve sorun alanını** tek bakışta çıkarabilirim:

| Senaryo | Sinyal | Aksiyon |
|---------|--------|---------|
| graph probe MF1 ≤ raw_concat + küçük delta | GNN zayıf | num_layers / edge_pathways / readout yeniden tasarlanmalı |
| sequence probe MF1 ≤ graph | Sequence decoder faydasız | gru_layers ↑ veya TCN kernel değiştirilmeli |
| only-spectral ≈ baseline | Waveform branch ölü | waveform_stem kapasitesi artırılmalı |
| only-waveform ≈ baseline | Spectral branch ölü | SpectralEncoder veya spectral features geliştirilmeli |
| σ(λ) ≈ 0 veya ≈ 1 | Fusion dejenere | `detach_gnn=False` dene ya da fusion'ı kaldır |
| bir katmanda dead% > 50 | Dead neurons | GELU↔ReLU değiştir, init rescale, LayerNorm ekle |
| herhangi bir yerde NaN/Inf | Sayısal patlama | grad_clip düşür, AMP kontrol et, init kontrol et |


## 9. Ablation Aggregate

Tüm tamamlanmış run'ların (`outputs/ablation/*.json`) karşılaştırmalı tablosu.

**Kullanım**: Toggle kombinasyonlarını değiştirerek notebook'u baştan sona
her biri için bir kez çalıştır (`USE_PATHWAY / USE_FUSION / USE_N1_MIXUP`
+ istenirse `n1_boost`, `weight_decay`, `label_smoothing`). Sonra bu
hücre tüm run'ları tek tabloda karşılaştırır ve ablation-ranking markdown'ı
üretir.


In [ ]:
# ============================================================
# Ablation aggregate table
# ------------------------------------------------------------
# outputs/ablation/*.json dosyalarını okur, HMM-post-processing
# MF1'e göre sıralar. Her run için toggle combo, test MF1/ACC/κ ve
# per-class F1'leri tek tabloda gösterir.
# ============================================================
import json
from pathlib import Path

ablation_dir = OUTPUTS_DIR / "ablation"
if not ablation_dir.exists():
    print(f"Henüz kayıt yok: {ablation_dir}")
else:
    records = []
    for p in sorted(ablation_dir.glob("*.json")):
        try:
            records.append(json.loads(p.read_text()))
        except Exception as e:
            print(f"[skip] {p.name}: {e}")

    if not records:
        print(f"Henüz kayıt yok: {ablation_dir}")
    else:
        # HMM MF1'e göre sırala (post-processing sonrası gerçek skorumuz)
        records.sort(key=lambda r: r["test"]["hmm"]["macro_f1"], reverse=True)

        print(f"\n{'Run':<40s} {'P':>2s} {'F':>2s} {'M':>2s} "
              f"{'Raw MF1':>8s} {'Bias MF1':>9s} {'HMM MF1':>8s} "
              f"{'HMM Acc':>8s} {'κ':>7s} {'F1_N1':>7s} {'F1_N2':>7s}")
        print("-" * 120)
        for r in records:
            t = r["toggles"]
            raw = r["test"]["raw"]
            bia = r["test"]["biased"]
            hmm = r["test"]["hmm"]
            print(
                f"{r['ablation_name']:<40s} "
                f"{'1' if t['pathway'] else '0':>2s} "
                f"{'1' if t['fusion']   else '0':>2s} "
                f"{'1' if t['n1_mixup'] else '0':>2s} "
                f"{raw['macro_f1']:>8.4f} {bia['macro_f1']:>9.4f} {hmm['macro_f1']:>8.4f} "
                f"{hmm['accuracy']:>8.4f} {hmm['kappa']:>7.4f} "
                f"{hmm['f1_N1']:>7.4f} {hmm['f1_N2']:>7.4f}"
            )

        # ΔMF1 analizi: baseline (hepsi OFF) varsa her toggle'ın izole
        # katkısını göster.
        def find_combo(pw: bool, fu: bool, mx: bool):
            for r in records:
                t = r["toggles"]
                if t["pathway"] == pw and t["fusion"] == fu and t["n1_mixup"] == mx:
                    return r
            return None

        base = find_combo(False, False, False)
        if base is not None:
            base_hmm = base["test"]["hmm"]["macro_f1"]
            print(f"\nBaseline (all OFF) HMM MF1 = {base_hmm:.4f}")
            for name, pw, fu, mx in [
                ("+pathway only",  True,  False, False),
                ("+fusion only",   False, True,  False),
                ("+mixup only",    False, False, True),
                ("+path+fusion",   True,  True,  False),
                ("+path+mixup",    True,  False, True),
                ("+fusion+mixup",  False, True,  True),
                ("all ON",         True,  True,  True),
            ]:
                r = find_combo(pw, fu, mx)
                if r is None:
                    continue
                dm = r["test"]["hmm"]["macro_f1"] - base_hmm
                print(f"  {name:<18s}: {r['test']['hmm']['macro_f1']:.4f}  "
                      f"(Δ{dm:+.4f})")

        # Markdown tablosu (makale için)
        print("\n\n### Markdown table\n")
        print("| Run | Pathway | Fusion | Mixup | Raw MF1 | Biased MF1 | HMM MF1 | HMM Acc | κ | F1_N1 |")
        print("|---|:-:|:-:|:-:|---:|---:|---:|---:|---:|---:|")
        for r in records:
            t = r["toggles"]
            hmm = r["test"]["hmm"]
            print(
                f"| {r['ablation_name']} "
                f"| {'✓' if t['pathway'] else '–'} "
                f"| {'✓' if t['fusion']   else '–'} "
                f"| {'✓' if t['n1_mixup'] else '–'} "
                f"| {r['test']['raw']['macro_f1']:.4f} "
                f"| {r['test']['biased']['macro_f1']:.4f} "
                f"| **{hmm['macro_f1']:.4f}** "
                f"| {hmm['accuracy']:.4f} "
                f"| {hmm['kappa']:.4f} "
                f"| {hmm['f1_N1']:.4f} |"
            )
